## Dataset Evaluation

In [1]:
"""
Pipeline: Preflight -> Conditional P(feature | gender) comparison
- Robust CSV reader (auto-detect delimiter)
- Light label n
ormalization (unify dashes/whitespace; case-insensitive compare)
- Saves all outputs to OUT_DIR
- BN sample columns (new): 
    gender, median_income, Age, Arrival_group, Business_size, 
    Industry, Institutional_sector, Job_duration, Visa_group
- Target CSVs used (others intentionally omitted):
    age_gender_transformed_v3.csv           -> Age
    arrivalGroup_gender_transformed.csv     -> Arrival_group
    businessSize_gender_transformed.csv     -> Business_size
    industry_gender_transformed_v3.csv      -> Industry
    sector_gender_transformed_v3.csv        -> Institutional_sector
    jobDuration_gender_transformed.csv      -> Job_duration
    visaGroup_gender_transformed.csv        -> Visa_group
"""

import os
import re
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

# =============== Robust CSV reader (auto-detect delimiter) ===============
def read_any(path: str) -> pd.DataFrame:
    """Read CSV; fallback to autodetected delimiter if collapsed to 1 column; strip headers."""
    path = os.path.expanduser(path)
    df = pd.read_csv(path)
    if df.shape[1] == 1:
        df = pd.read_csv(path, sep=None, engine="python")
    df.columns = [c.strip() for c in df.columns]
    return df

# =============== Light label normalization helpers ===============
_DASH_RX = re.compile(r"[\u2012\u2013\u2014\u2015\u2212]")  # figure dash, en/em dash, minus
_WS_RX = re.compile(r"\s+")

def normalize_token(x: str) -> str:
    """
    Normalize a category token for matching:
    - strip
    - unify various dashes to hyphen '-'
    - collapse multiple spaces to single space
    - lowercase (for stable exact-compare)
    """
    s = str(x).strip()
    s = _DASH_RX.sub("-", s)
    s = _WS_RX.sub(" ", s)
    return s.lower()

def norm_gender(series: pd.Series) -> pd.Series:
    """Normalize gender labels to {FEMALES, MALES} (uppercase)."""
    mp = {"female": "FEMALES", "females": "FEMALES", "f": "FEMALES",
          "male": "MALES", "males": "MALES", "m": "MALES"}
    def _one(x):
        if pd.isna(x):
            return x
        t = str(x).strip()
        return mp.get(t.lower(), t.upper())
    return series.astype(object).map(_one)

def hellinger(p: np.ndarray, q: np.ndarray) -> float:
    p = np.clip(p, 0, 1); q = np.clip(q, 0, 1)
    return float((1/np.sqrt(2)) * np.linalg.norm(np.sqrt(p) - np.sqrt(q)))

def tvd(p: np.ndarray, q: np.ndarray) -> float:
    return float(0.5 * np.abs(p - q).sum())

# =============== Target CSV -> BN sample column map (EXPLICIT) ===============
TABLE_COLUMN_MAP: Dict[str, str] = {
    "age_gender_transformed_v3.csv":          "Age",
    "arrivalGroup_gender_transformed.csv":    "Arrival_group",
    "businessSize_gender_transformed.csv":    "Business_size",
    "industry_gender_transformed_v3.csv":     "Industry",
    "sector_gender_transformed_v3.csv":       "Institutional_sector",
    "jobDuration_gender_transformed.csv":     "Job_duration",
    "visaGroup_gender_transformed.csv":       "Visa_group",
}

# =============== Overlap helper (for preflight; uses normalized tokens) ===============
def most_overlap_column(sample_df: pd.DataFrame,
                        target_values: List[str],
                        candidate_cols: Optional[List[str]] = None,
                        min_overlap_ratio: float = 0.3) -> Tuple[Optional[str], float, int]:
    """
    Return (best_column, overlap_ratio, n_missing_target_values) 
    by comparing normalized strings (case-insensitive, dash/whitespace unified).
    """
    if candidate_cols is None:
        candidate_cols = [
            c for c in sample_df.columns
            if (sample_df[c].dtype == "object"
                or str(sample_df[c].dtype).startswith("category")
                or sample_df[c].nunique() < 200)
        ]
    target_norm = {normalize_token(v) for v in target_values}
    best_col, best_overlap, best_missing = None, -1.0, len(target_norm)

    for c in candidate_cols:
        svals = sample_df[c].dropna().astype(str).map(normalize_token).unique().tolist()
        sset = set(svals)
        inter = len(target_norm.intersection(sset))
        ratio = inter / max(1, len(target_norm))
        missing = len(target_norm - sset)
        if ratio > best_overlap:
            best_overlap, best_col, best_missing = ratio, c, missing

    if best_overlap >= min_overlap_ratio:
        return best_col, best_overlap, best_missing
    return None, best_overlap, best_missing

# =============== Preflight (structure & mapping) ===============
def preflight_check(sample_path: str,
                    target_paths: List[str],
                    gender_col_candidates: List[str] = ["gender", "Gender", "GENDER"]
                   ) -> pd.DataFrame:
    """
    For each target CSV:
      - validate required columns: gender / feature / PROPORTION
      - check mapping via explicit TABLE_COLUMN_MAP:
          * OK if the mapped sample column exists
          * FAIL if explicit column missing in BN sample
      - (fallback) auto-overlap match if no explicit map (not used here)
      - report overlap ratio & missing target categories when auto-matching is used
      - status: OK / WARN / FAIL with issue text
    """
    sdf = read_any(sample_path)
    sdf.columns = [c.strip() for c in sdf.columns]

    # locate gender column in BN sample
    s_gender_col = None
    for c in gender_col_candidates:
        if c in sdf.columns:
            s_gender_col = c; break
    if s_gender_col is None:
        for c in sdf.columns:
            if c.lower() == "gender":
                s_gender_col = c; break

    rows = []
    for p in target_paths:
        name = os.path.basename(p)
        try:
            tdf = read_any(p)
        except Exception as e:
            rows.append({"table": name, "status": "FAIL", "issue": f"cannot read CSV: {e}"})
            continue

        cols_lower = {c.lower(): c for c in tdf.columns}
        gcol = cols_lower.get("gender"); fcol = cols_lower.get("feature")
        pcol = None
        for k in tdf.columns:
            if k.strip().lower() in ["proportion", "prop", "share"]:
                pcol = k; break

        if not (gcol and fcol and pcol):
            rows.append({
                "table": name, "status": "FAIL",
                "issue": "missing required columns: gender/feature/PROPORTION",
                "bn_feature_col": None, "overlap_ratio": None, "missing_targets": None
            })
            continue

        if s_gender_col is None:
            rows.append({"table": name, "status": "FAIL",
                         "issue": "BN sample missing a gender column", "bn_feature_col": None})
            continue

        # explicit map (used here)
        explicit = TABLE_COLUMN_MAP.get(name)
        if explicit is not None:
            if explicit in sdf.columns:
                rows.append({
                    "table": name, "status": "OK",
                    "issue": "explicit TABLE_COLUMN_MAP used",
                    "bn_feature_col": explicit, "overlap_ratio": None, "missing_targets": None
                })
                continue
            else:
                rows.append({
                    "table": name, "status": "FAIL",
                    "issue": f"TABLE_COLUMN_MAP column '{explicit}' not in BN sample",
                    "bn_feature_col": None
                })
                continue

        # fallback: auto-overlap (shouldn't trigger with our explicit map defined)
        tvals = sorted(tdf[fcol].astype(str).str.strip().unique().tolist())
        best_col, ratio, missing = most_overlap_column(sdf, tvals)
        if best_col is None:
            rows.append({
                "table": name, "status": "FAIL",
                "issue": "auto-match failed: low overlap or no matching column",
                "overlap_ratio": float(ratio), "missing_targets": int(missing)
            })
        else:
            status = "OK" if ratio >= 0.5 else "WARN"
            rows.append({
                "table": name, "status": status,
                "issue": "auto-match" if status == "OK" else "auto-match (overlap < 0.5; review)",
                "bn_feature_col": best_col, "overlap_ratio": float(ratio),
                "missing_targets": int(missing)
            })
    return pd.DataFrame(rows)

# =============== Conditional comparison helpers (use normalized tokens) ===============
def sample_conditional(sample_df: pd.DataFrame, gender_col: str, feat_col: str) -> pd.DataFrame:
    """Compute P(feature | gender) from the BN sample using normalized category tokens."""
    s = sample_df.copy()
    s[gender_col] = norm_gender(s[gender_col])
    s[feat_col]   = s[feat_col].astype(str).map(normalize_token)

    cnt = s.groupby([gender_col, feat_col], as_index=False).size()
    den = s.groupby(gender_col, as_index=False).size().rename(columns={"size": "den"})
    out = cnt.merge(den, on=gender_col, how="left")
    out["sample_prop"] = out["size"] / out["den"].replace(0, np.nan)
    out = out.rename(columns={gender_col: "gender", feat_col: "feature"})[["gender", "feature", "sample_prop"]]
    return out

def target_to_conditional(target_df: pd.DataFrame,
                          sample_enum: List[str]) -> Tuple[pd.DataFrame, Dict[str, int]]:
    """
    Convert target joint P(g, c) into conditional P(c | g).
    Compare only on the intersection with sample_enum (normalized);
    report how many target categories were dropped.
    """
    df = target_df.copy()
    gcol = next(c for c in df.columns if c.lower() == "gender")
    fcol = next(c for c in df.columns if c.lower() == "feature")
    pcol = next(c for c in df.columns if c.lower() in ["proportion", "prop", "share"])

    df[gcol] = norm_gender(df[gcol])
    df[fcol] = df[fcol].astype(str).map(normalize_token)
    df[pcol] = df[pcol].astype(float)

    keep = df[df[fcol].isin(sample_enum)].copy()
    dropped = int((~df[fcol].isin(sample_enum)).sum())

    cond = (keep.groupby([gcol, fcol], as_index=False)[pcol].sum()
              .rename(columns={gcol: "gender", fcol: "feature", pcol: "target_prop"}))
    cond["target_prop"] = cond.groupby("gender")["target_prop"].transform(
        lambda x: x / (x.sum() if x.sum() else 1.0)
    )

    # ensure full support per gender (fill missing categories with 0, then renormalize)
    rows = []
    for g in cond["gender"].dropna().unique():
        sub = cond[cond["gender"] == g].set_index("feature")["target_prop"]
        sub = sub.reindex(sample_enum).fillna(0.0)
        sub = sub / (sub.sum() if sub.sum() else 1.0)
        rows += [{"gender": g, "feature": k, "target_prop": v} for k, v in sub.items()]
    cond = pd.DataFrame(rows)
    return cond, {"dropped_unmapped": dropped}

def conditional_compare(sample_path: str,
                        target_paths: List[str],
                        table_col_map: Dict[str, str],
                        out_dir: Optional[str] = None) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Compare P(feature|gender) between targets and BN sample for all provided tables.
    Save summary/details to out_dir if provided.
    """
    sdf = read_any(sample_path)
    sdf.columns = [c.strip() for c in sdf.columns]
    gender_col = next(c for c in sdf.columns if c.lower() == "gender")

    summary_rows, detail_rows = [], []
    for tpath in target_paths:
        name = os.path.basename(tpath)
        if name not in table_col_map:
            summary_rows.append({"table": name, "error": "TABLE_COLUMN_MAP missing"})
            continue
        feat_col = table_col_map[name]
        if feat_col not in sdf.columns:
            summary_rows.append({"table": name, "error": f"BN sample missing column: {feat_col}"})
            continue

        tdf = read_any(tpath)

        # sample enumeration on NORMALIZED tokens
        sample_enum = sorted(
            sdf[feat_col].dropna().astype(str).map(normalize_token).unique().tolist()
        )

        # Build P(c|g) for target and sample
        t_cond, notes = target_to_conditional(tdf, sample_enum)
        s_cond = sample_conditional(sdf[[gender_col, feat_col]], gender_col, feat_col)

        merged = (t_cond.merge(s_cond, on=["gender", "feature"], how="outer")
                         .fillna(0.0).sort_values(["gender", "feature"]))

        # Distances per gender; aggregate to mean/max
        genders = merged["gender"].unique().tolist()
        tvds, hels, maxdevs = [], [], []
        for g in genders:
            uu = merged[merged["gender"] == g]
            P = uu["target_prop"].to_numpy(float)
            Q = uu["sample_prop"].to_numpy(float)
            tvds.append(tvd(P, Q))
            hels.append(hellinger(P, Q))
            maxdevs.append(float(np.max(np.abs(P - Q))) if len(P) else np.nan)

        summary_rows.append({
            "table": name,
            "bn_feature_col": feat_col,
            "tvd_mean": float(np.nanmean(tvds)),
            "tvd_max": float(np.nanmax(tvds)),
            "hellinger_mean": float(np.nanmean(hels)),
            "hellinger_max": float(np.nanmax(hels)),
            "max_abs_dev_mean": float(np.nanmean(maxdevs)),
            "max_abs_dev_max": float(np.nanmax(maxdevs)),
            "notes": f"dropped_unmapped={notes.get('dropped_unmapped',0)}; sample_cats={len(sample_enum)}",
        })

        for _, r in merged.iterrows():
            detail_rows.append({
                "table": name, "bn_feature_col": feat_col,
                "gender": r["gender"], "feature": r["feature"],
                "target_prop": float(r["target_prop"]),
                "sample_prop": float(r["sample_prop"]),
                "diff_pp": float((r["sample_prop"] - r["target_prop"]) * 100.0),
                "abs_diff": float(abs(r["sample_prop"] - r["target_prop"]))
            })

    cond_summary = pd.DataFrame(summary_rows)
    cond_details = pd.DataFrame(detail_rows)

    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
        cond_summary.to_csv(os.path.join(out_dir, "conditional_compare_summary.csv"), index=False)
        cond_details.to_csv(os.path.join(out_dir, "conditional_compare_details.csv"), index=False)

    return cond_summary, cond_details

# =============== Orchestrator: Preflight -> Conditional Compare ===============
def run_with_preflight_and_conditional(sample_path: str,
                                       target_prefix: str,
                                       target_files: List[str],
                                       table_col_map: Dict[str, str],
                                       out_dir: str) -> Dict[str, str]:
    """
    1) Run preflight_check and save 'preflight_report.csv'
    2) Run conditional compare and save:
       - 'conditional_compare_summary.csv'
       - 'conditional_compare_details.csv'
    Returns dict of output file paths.
    """
    sample_path = os.path.expanduser(sample_path)
    target_prefix = os.path.expanduser(target_prefix)
    target_paths = [os.path.join(target_prefix, fn) for fn in target_files]

    missing = [p for p in [sample_path, *target_paths] if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError("Missing files:\n" + "\n".join(missing))

    os.makedirs(out_dir, exist_ok=True)

    # 1) Preflight
    pre = preflight_check(sample_path, target_paths)
    pre_path = os.path.join(out_dir, "preflight_report.csv")
    pre.to_csv(pre_path, index=False)
    print(f"[Preflight] report saved to: {pre_path}")

    # 2) Conditional comparison
    cond_sum, cond_det = conditional_compare(sample_path, target_paths, table_col_map, out_dir=out_dir)
    sum_path = os.path.join(out_dir, "conditional_compare_summary.csv")
    det_path = os.path.join(out_dir, "conditional_compare_details.csv")
    print(f"[Conditional] summary saved to: {sum_path}")
    print(f"[Conditional] details saved to: {det_path}")

    return {"preflight": pre_path, "conditional_summary": sum_path, "conditional_details": det_path}

SAMPLE = "~/Desktop/DataScience_Project_Team_25/data/synthetic/assuming_independent_initial_synthetic.csv"
TARGET_PREFIX = "~/Desktop/DataScience_Project_Team_25/data/marginal_tables_transformed"
TARGET_FILES = [
     "age_gender_transformed_v3.csv",
     "arrivalGroup_gender_transformed.csv",
     "businessSize_gender_transformed.csv",
     "industry_gender_transformed_v3.csv",
     "sector_gender_transformed_v3.csv",
     "jobDuration_gender_transformed.csv",
     "visaGroup_gender_transformed.csv",
 ]
OUT_DIR = "./eval_outputs_v3"
outputs = run_with_preflight_and_conditional(SAMPLE, TARGET_PREFIX, TARGET_FILES, TABLE_COLUMN_MAP, OUT_DIR)
outputs  

[Preflight] report saved to: ./eval_outputs_v3/preflight_report.csv
[Conditional] summary saved to: ./eval_outputs_v3/conditional_compare_summary.csv
[Conditional] details saved to: ./eval_outputs_v3/conditional_compare_details.csv


{'preflight': './eval_outputs_v3/preflight_report.csv',
 'conditional_summary': './eval_outputs_v3/conditional_compare_summary.csv',
 'conditional_details': './eval_outputs_v3/conditional_compare_details.csv'}

In [3]:
# ==== Final drilldown summary (what to drill and why) ====
import os
import numpy as np
import pandas as pd

def load_eval_outputs(out_dir: str):
    pre  = pd.read_csv(os.path.join(out_dir, "preflight_report.csv"))
    summ = pd.read_csv(os.path.join(out_dir, "conditional_compare_summary.csv"))
    deta = pd.read_csv(os.path.join(out_dir, "conditional_compare_details.csv"))
    return pre, summ, deta

def pick_tables_for_drilldown(cond_summary: pd.DataFrame,
                              tvd_mean_thr: float = 0.06,
                              max_abs_dev_thr: float = 0.08) -> pd.DataFrame:
    """Flag tables that should be drilled: TVD mean or Max abs deviation beyond thresholds."""
    df = cond_summary.copy()
    # ensure numeric
    for col in ["tvd_mean", "max_abs_dev_max"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df["flag_tvd_mean"] = df["tvd_mean"] > tvd_mean_thr
    df["flag_max_dev"]  = df["max_abs_dev_max"] > max_abs_dev_thr
    df["needs_drilldown"] = df[["flag_tvd_mean", "flag_max_dev"]].any(axis=1)
    # human-readable reason
    part_a = np.where(df["flag_tvd_mean"], f"tvd_mean>{tvd_mean_thr}", "")
    part_b = np.where(df["flag_max_dev"],  f"max_abs_dev_max>{max_abs_dev_thr}", "")
    df["reason"] = [" & ".join([a, b]).strip(" & ") for a, b in zip(part_a, part_b)]
    return df

def _table_detail_diagnostics(details: pd.DataFrame, eps: float = 1e-9) -> pd.DataFrame:
    """
    Aggregates detail-level diagnostics to table-level heuristics:
      - zero_coverage_rate: share of rows with target>0 & sample==0
      - only_in_sample_rate: share of rows with target==0 & sample>0
      - direction_bias: share of rows with |delta|>tiny that go in the dominant direction (max(over, under))
    """
    d = details.copy()
    d["delta"] = d["sample_prop"] - d["target_prop"]
    d["over"]  = (d["delta"] >  0 + eps)
    d["under"] = (d["delta"] <  0 - eps)
    d["sig"]   = (d["delta"].abs() > eps)
    d["zero_coverage"]   = (d["target_prop"] > eps) & (d["sample_prop"] <= eps)
    d["only_in_sample"]  = (d["target_prop"] <= eps) & (d["sample_prop"] >  eps)

    grp = d.groupby("table", as_index=False).agg(
        n_rows=("table", "size"),
        zero_cov=("zero_coverage", "sum"),
        only_in=("only_in_sample", "sum"),
        over_sig=("over", "sum"),
        under_sig=("under", "sum"),
        sig_cnt=("sig", "sum"),
        mean_abs_diff=("abs_diff", "mean"),
        max_abs_diff=("abs_diff", "max"),
    )
    grp["zero_coverage_rate"]  = grp["zero_cov"] / grp["n_rows"].replace(0, np.nan)
    grp["only_in_sample_rate"] = grp["only_in"]  / grp["n_rows"].replace(0, np.nan)
    # direction bias among significant rows
    grp["direction_bias"] = np.where(
        grp["sig_cnt"] > 0,
        np.maximum(grp["over_sig"], grp["under_sig"]) / grp["sig_cnt"],
        0.0
    )
    return grp[[
        "table", "n_rows",
        "zero_coverage_rate", "only_in_sample_rate", "direction_bias",
        "mean_abs_diff", "max_abs_diff"
    ]]

def _suggest_action_row(pre_status: str,
                        zero_rate: float,
                        only_rate: float,
                        direction_bias: float,
                        tvd_mean: float,
                        max_abs_dev: float) -> str:
    """
    Heuristic suggestion:
      - FAIL in preflight -> fix mapping/columns first
      - high only_in_sample_rate -> label alignment (enumeration/normalization)
      - high zero_coverage_rate -> BN rare support or quotas
      - strong direction bias -> calibration/raking P(feature|gender)
      - otherwise -> light calibration/fine-tune
    """
    if str(pre_status).upper() == "FAIL":
        return "fix_mapping_or_missing_columns"
    if only_rate is not None and only_rate >= 0.10:
        return "label_alignment_normalization"
    if zero_rate is not None and zero_rate >= 0.10:
        return "BN_support_or_sampling_quotas_for_rare_categories"
    if direction_bias is not None and direction_bias >= 0.70:
        return "calibration_raking_within_gender"
    if (tvd_mean is not None and tvd_mean > 0.05) or (max_abs_dev is not None and max_abs_dev > 0.08):
        return "calibration_raking_within_gender"
    return "light_calibration_or_leave_as_is"

def _priority_row(pre_status: str,
                  tvd_mean: float,
                  max_abs_dev: float) -> str:
    """Simple triage priority."""
    if str(pre_status).upper() == "FAIL":
        return "High"
    if (tvd_mean is not None and tvd_mean > 0.08) or (max_abs_dev is not None and max_abs_dev > 0.10):
        return "High"
    if (tvd_mean is not None and tvd_mean > 0.05) or (max_abs_dev is not None and max_abs_dev > 0.08):
        return "Medium"
    return "Low"

def summarize_drill_candidates(out_dir: str,
                               tvd_mean_thr: float = 0.06,
                               max_abs_dev_thr: float = 0.08,
                               save_name: str = "drilldown_summary.csv") -> pd.DataFrame:
    """
    Produce a single CSV that tells:
      - which tables need drilldown
      - which metrics triggered it (reason)
      - preflight status/issue
      - diagnostic rates and suggested action
      - priority
    Saves to OUT_DIR/save_name and returns the DataFrame.
    """
    pre, summ, deta = load_eval_outputs(out_dir)

    # Ensure numeric columns in summary
    for c in ["tvd_mean", "tvd_max", "hellinger_mean", "hellinger_max", "max_abs_dev_mean", "max_abs_dev_max"]:
        if c in summ.columns:
            summ[c] = pd.to_numeric(summ[c], errors="coerce")

    # 1) pick by thresholds
    picks = pick_tables_for_drilldown(summ, tvd_mean_thr, max_abs_dev_thr)

    # 2) aggregate detail diagnostics per table
    diag = _table_detail_diagnostics(deta)

    # 3) merge preflight info (take latest status/issue per table)
    pre_min = pre.groupby("table", as_index=False).agg(
        preflight_status=("status", "first"),
        preflight_issue=("issue", "first")
    )

    # 4) merge all
    out = (picks.merge(pre_min, on="table", how="left")
                .merge(diag, on="table", how="left"))

    # 5) suggestion + priority
    out["suggested_action"] = [
        _suggest_action_row(
            pre_status=row.get("preflight_status"),
            zero_rate=row.get("zero_coverage_rate"),
            only_rate=row.get("only_in_sample_rate"),
            direction_bias=row.get("direction_bias"),
            tvd_mean=row.get("tvd_mean"),
            max_abs_dev=row.get("max_abs_dev_max"),
        )
        for _, row in out.iterrows()
    ]
    out["priority"] = [
        _priority_row(
            pre_status=row.get("preflight_status"),
            tvd_mean=row.get("tvd_mean"),
            max_abs_dev=row.get("max_abs_dev_max"),
        )
        for _, row in out.iterrows()
    ]

    # reorder columns
    cols = [
        "table", "bn_feature_col",
        "preflight_status", "preflight_issue",
        "tvd_mean", "max_abs_dev_max", "hellinger_mean",
        "needs_drilldown", "reason",
        "zero_coverage_rate", "only_in_sample_rate", "direction_bias",
        "suggested_action", "priority",
        "notes"
    ]
    cols = [c for c in cols if c in out.columns]
    out = out[cols].sort_values(["priority", "tvd_mean", "max_abs_dev_max"], ascending=[True, False, False])

    # save
    path = os.path.join(out_dir, save_name)
    out.to_csv(path, index=False)
    print(f"[Final] drilldown summary saved to: {path}")
    # also print a concise view to screen
    display_cols = [c for c in ["table", "preflight_status", "tvd_mean", "max_abs_dev_max",
                                "needs_drilldown", "reason", "suggested_action", "priority"] if c in out.columns]
    print(out[display_cols].to_string(index=False))
    return out

OUT_DIR = "./eval_outputs_v3"
summarize_drill_candidates(OUT_DIR, tvd_mean_thr=0.06, max_abs_dev_thr=0.08)

[Final] drilldown summary saved to: ./eval_outputs_v3/drilldown_summary.csv
                              table preflight_status  tvd_mean  max_abs_dev_max  needs_drilldown                               reason                 suggested_action priority
   sector_gender_transformed_v3.csv               OK  0.274498         0.300602             True tvd_mean>0.06 & max_abs_dev_max>0.08    label_alignment_normalization     High
 industry_gender_transformed_v3.csv               OK  0.209199         0.078995             True                        tvd_mean>0.06 calibration_raking_within_gender     High
      age_gender_transformed_v3.csv               OK  0.138987         0.054862             True                        tvd_mean>0.06 calibration_raking_within_gender     High
arrivalGroup_gender_transformed.csv               OK  0.102707         0.105229             True tvd_mean>0.06 & max_abs_dev_max>0.08 calibration_raking_within_gender     High
   visaGroup_gender_transformed.csv         

,table,bn_feature_col,preflight_status,preflight_issue,tvd_mean,max_abs_dev_max,hellinger_mean,needs_drilldown,reason,zero_coverage_rate,only_in_sample_rate,direction_bias,suggested_action,priority,notes
4,sector_gender_transformed_v3.csv,Institutional_sector,OK,explicit TABLE_COLUMN_MAP used,0.274498,0.300602,0.375179,True,tvd_mean>0.06 & max_abs_dev_max>0.08,0.0,0.400000,0.600000,label_alignment_normalization,High,dropped_unmapped=2; sample_cats=5
3,industry_gender_transformed_v3.csv,Industry,OK,explicit TABLE_COLUMN_MAP used,0.209199,0.078995,0.232428,True,tvd_mean>0.06,0.0,0.076923,0.512821,calibration_raking_within_gender,High,dropped_unmapped=0; sample_cats=19
0,age_gender_transformed_v3.csv,Age,OK,explicit TABLE_COLUMN_MAP used,0.138987,0.054862,0.128585,True,tvd_mean>0.06,0.0,0.000000,0.500000,calibration_raking_within_gender,High,dropped_unmapped=0; sample_cats=17
1,arrivalGroup_gender_transformed.csv,Arrival_group,OK,explicit TABLE_COLUMN_MAP used,0.102707,0.105229,0.073485,True,tvd_mean>0.06 & max_abs_dev_max>0.08,0.0,0.000000,0.500000,calibration_raking_within_gender,High,dropped_unmapped=0; sample_cats=3
6,visaGroup_gender_transformed.csv,Visa_group,OK,explicit TABLE_COLUMN_MAP used,0.097575,0.062520,0.106012,True,tvd_mean>0.06,0.0,0.000000,0.535714,calibration_raking_within_gender,High,dropped_unmapped=0; sample_cats=14
2,businessSize_gender_transformed.csv,Business_size,OK,explicit TABLE_COLUMN_MAP used,0.070009,0.072586,0.054055,True,tvd_mean>0.06,0.0,0.000000,0.500000,calibration_raking_within_gender,Medium,dropped_unmapped=0; sample_cats=4
5,jobDuration_gender_transformed.csv,Job_duration,OK,explicit TABLE_COLUMN_MAP used,0.050681,0.052006,0.042099,False,,0.0,0.000000,0.500000,calibration_raking_within_gender,Medium,dropped_unmapped=0; sample_cats=4


In [6]:
"""
Drill-down generator
- Consumes the files produced by the previous pipeline:
    OUT_DIR/preflight_report.csv
    OUT_DIR/conditional_compare_summary.csv
    OUT_DIR/conditional_compare_details.csv
- Selects tables that need drill-down based on thresholds
- Emits per-table drill-down CSVs under OUT_DIR/drilldown/
"""

import os
import pandas as pd
import numpy as np

def load_eval_outputs(out_dir: str):
    """Load preflight, conditional summary/details; return dataframes."""
    pre_path  = os.path.join(out_dir, "preflight_report.csv")
    sum_path  = os.path.join(out_dir, "conditional_compare_summary.csv")
    det_path  = os.path.join(out_dir, "conditional_compare_details.csv")
    for p in [pre_path, sum_path, det_path]:
        if not os.path.exists(p):
            raise FileNotFoundError(f"Missing file: {p}")
    pre  = pd.read_csv(pre_path)
    summ = pd.read_csv(sum_path)
    deta = pd.read_csv(det_path)
    return pre, summ, deta

def pick_tables_for_drilldown(
    cond_summary: pd.DataFrame,
    tvd_mean_thr: float = 0.06,
    max_abs_dev_thr: float = 0.08
) -> pd.DataFrame:
    """
    Decide which tables should be drilled down.
    - Flag a table if tvd_mean > tvd_mean_thr OR max_abs_dev_max > max_abs_dev_thr
    - Build a human-readable 'reason' without NumPy string ops
    """
    df = cond_summary.copy()

    # Ensure numeric types (avoid 'object' strings breaking comparisons)
    for col in ["tvd_mean", "max_abs_dev_max"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df["flag_tvd_mean"] = df["tvd_mean"] > tvd_mean_thr
    df["flag_max_dev"]  = df["max_abs_dev_max"] > max_abs_dev_thr
    df["needs_drilldown"] = df[["flag_tvd_mean", "flag_max_dev"]].any(axis=1)

    # Build reason using plain Python join over per-row parts
    part_a = np.where(df["flag_tvd_mean"], f"tvd_mean>{tvd_mean_thr}", "")
    part_b = np.where(df["flag_max_dev"], f"max_abs_dev_max>{max_abs_dev_thr}", "")
    reasons = [" & ".join([a, b]).strip(" & ") for a, b in zip(part_a, part_b)]
    df["reason"] = reasons

    return df

def _add_diagnostics(detail: pd.DataFrame) -> pd.DataFrame:
    """
    Add diagnostic flags to conditional_compare_details:
    - sign: 'over' if sample>target, 'under' if sample<target, 'equal' if ~0
    - zero_coverage: target>0 but sample==0
    - only_in_sample: target==0 but sample>0
    """
    d = detail.copy()
    d["delta"] = d["sample_prop"] - d["target_prop"]
    d["sign"] = np.where(
        np.isclose(d["delta"], 0.0, atol=1e-12), "equal",
        np.where(d["delta"] > 0, "over", "under")
    )
    d["zero_coverage"] = (d["target_prop"] > 0) & (d["sample_prop"] == 0)
    d["only_in_sample"] = (d["target_prop"] == 0) & (d["sample_prop"] > 0)
    # convenience: percentage points columns rounded for readability
    d["diff_pp"]  = (d["diff_pp"]).round(3)      # already in percentage points
    d["abs_diff"] = (d["abs_diff"]).round(6)     # as proportion
    return d

def make_drilldown_files(
    out_dir: str,
    cond_summary: pd.DataFrame,
    cond_details: pd.DataFrame,
    tvd_mean_thr: float = 0.06,
    max_abs_dev_thr: float = 0.08,
    topk: int = 10
) -> dict:
    """
    Create drill-down CSVs under OUT_DIR/drilldown/.
    Returns a dict with:
      - index: path to drilldown_index.csv
      - tables: list of {table, by_gender_path, topk_path}
    """
    drill_dir = os.path.join(out_dir, "drilldown")
    os.makedirs(drill_dir, exist_ok=True)

    # 1) select tables to drill
    picks = pick_tables_for_drilldown(cond_summary, tvd_mean_thr, max_abs_dev_thr)
    picks_path = os.path.join(drill_dir, "drilldown_index.csv")
    picks.to_csv(picks_path, index=False)
    print(f"[Drilldown] index saved to: {picks_path}")

    # 2) build per-table drilldowns
    details = _add_diagnostics(cond_details)
    results = []
    for _, row in picks[picks["needs_drilldown"]].iterrows():
        table = row["table"]
        sub = details[details["table"] == table].copy()

        # (A) per-gender sorted by |diff| desc
        #     keep useful columns and order
        by_gender = []
        for g in sorted(sub["gender"].dropna().unique()):
            gg = sub[sub["gender"] == g].copy()
            gg["abs_diff_rank"] = gg["abs_diff"].rank(method="first", ascending=False)
            gg = gg.sort_values("abs_diff", ascending=False)
            by_gender.append(gg)
        by_gender = pd.concat(by_gender, ignore_index=True)
        by_gender = by_gender[[
            "table", "bn_feature_col", "gender", "feature",
            "target_prop", "sample_prop", "diff_pp", "abs_diff",
            "sign", "zero_coverage", "only_in_sample", "abs_diff_rank"
        ]]

        by_gender_path = os.path.join(drill_dir, f"{os.path.splitext(table)[0]}_by_gender.csv")
        by_gender.to_csv(by_gender_path, index=False)

        # (B) top-K per gender (largest absolute deviations)
        topk_rows = []
        for g in sorted(sub["gender"].dropna().unique()):
            gg = sub[sub["gender"] == g].sort_values("abs_diff", ascending=False).head(topk).copy()
            gg["rank_in_gender"] = range(1, len(gg) + 1)
            topk_rows.append(gg)
        topk_df = pd.concat(topk_rows, ignore_index=True)
        topk_df = topk_df[[
            "table", "bn_feature_col", "gender", "feature",
            "target_prop", "sample_prop", "diff_pp", "abs_diff",
            "sign", "zero_coverage", "only_in_sample", "rank_in_gender"
        ]]
        topk_path = os.path.join(drill_dir, f"{os.path.splitext(table)[0]}_top{topk}.csv")
        topk_df.to_csv(topk_path, index=False)

        print(f"[Drilldown] {table}: by_gender -> {by_gender_path}")
        print(f"[Drilldown] {table}: top{topk}    -> {topk_path}")

        results.append({"table": table, "by_gender": by_gender_path, "topk": topk_path})

    return {"index": picks_path, "tables": results}


OUT_DIR = "./eval_outputs_v3"  # same dir you used previously
pre, cond_sum, cond_det = load_eval_outputs(OUT_DIR)
drill_outputs = make_drilldown_files(
OUT_DIR,
cond_summary=cond_sum,
cond_details=cond_det,
tvd_mean_thr=0.06,        # triage threshold (mean TVD)
max_abs_dev_thr=0.08,     # triage threshold (any single category deviation)
topk=10                   # how many largest deviations per gender to surface
 )
drill_outputs

[Drilldown] index saved to: ./eval_outputs_v3/drilldown/drilldown_index.csv
[Drilldown] age_gender_transformed_v3.csv: by_gender -> ./eval_outputs_v3/drilldown/age_gender_transformed_v3_by_gender.csv
[Drilldown] age_gender_transformed_v3.csv: top10    -> ./eval_outputs_v3/drilldown/age_gender_transformed_v3_top10.csv
[Drilldown] arrivalGroup_gender_transformed.csv: by_gender -> ./eval_outputs_v3/drilldown/arrivalGroup_gender_transformed_by_gender.csv
[Drilldown] arrivalGroup_gender_transformed.csv: top10    -> ./eval_outputs_v3/drilldown/arrivalGroup_gender_transformed_top10.csv
[Drilldown] businessSize_gender_transformed.csv: by_gender -> ./eval_outputs_v3/drilldown/businessSize_gender_transformed_by_gender.csv
[Drilldown] businessSize_gender_transformed.csv: top10    -> ./eval_outputs_v3/drilldown/businessSize_gender_transformed_top10.csv
[Drilldown] industry_gender_transformed_v3.csv: by_gender -> ./eval_outputs_v3/drilldown/industry_gender_transformed_v3_by_gender.csv
[Drilldown] i

{'index': './eval_outputs_v3/drilldown/drilldown_index.csv',
 'tables': [{'table': 'age_gender_transformed_v3.csv',
   'by_gender': './eval_outputs_v3/drilldown/age_gender_transformed_v3_by_gender.csv',
   'topk': './eval_outputs_v3/drilldown/age_gender_transformed_v3_top10.csv'},
  {'table': 'arrivalGroup_gender_transformed.csv',
   'by_gender': './eval_outputs_v3/drilldown/arrivalGroup_gender_transformed_by_gender.csv',
   'topk': './eval_outputs_v3/drilldown/arrivalGroup_gender_transformed_top10.csv'},
  {'table': 'businessSize_gender_transformed.csv',
   'by_gender': './eval_outputs_v3/drilldown/businessSize_gender_transformed_by_gender.csv',
   'topk': './eval_outputs_v3/drilldown/businessSize_gender_transformed_top10.csv'},
  {'table': 'industry_gender_transformed_v3.csv',
   'by_gender': './eval_outputs_v3/drilldown/industry_gender_transformed_v3_by_gender.csv',
   'topk': './eval_outputs_v3/drilldown/industry_gender_transformed_v3_top10.csv'},
  {'table': 'sector_gender_transfo

In [12]:
# === Auto Action Planner: from your existing outputs to a single modification checklist ===
# Inputs:
#   OUT_DIR/preflight_report.csv
#   OUT_DIR/conditional_compare_summary.csv
#   OUT_DIR/conditional_compare_details.csv
# Outputs:
#   OUT_DIR/modification_plan.csv   (fine-grained action items)
#   OUT_DIR/exec_summary.csv        (table-level summary & priority)

import os
import numpy as np
import pandas as pd

# -------------------------
# Config (you can tweak)
# -------------------------
OUT_DIR = "./eval_outputs_v3"
# triage thresholds (must match how you picked drilldown)
TVD_MEAN_THR     = 0.06
MAX_ABS_DEV_THR  = 0.08
# action selection thresholds
TOPK_PER_GENDER  = 10      # how many largest deviations per gender to propose
MIN_ABS_DIFF_PCT = 0.003   # ignore tiny diffs (<0.3 percentage points); this is on proportion scale
RESIDUAL_MIN_PP  = 0.2     # if residual (pp) per gender is between -0.2 and 0.2, skip residual row

# -------------------------
# Load inputs
# -------------------------
def _must(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")
    return path

pre_path  = _must(os.path.join(OUT_DIR, "preflight_report.csv"))
sum_path  = _must(os.path.join(OUT_DIR, "conditional_compare_summary.csv"))
det_path  = _must(os.path.join(OUT_DIR, "conditional_compare_details.csv"))

pre  = pd.read_csv(pre_path)
summ = pd.read_csv(sum_path)
deta = pd.read_csv(det_path)

# ensure numeric for summary
for c in ["tvd_mean","tvd_max","hellinger_mean","hellinger_max","max_abs_dev_mean","max_abs_dev_max"]:
    if c in summ.columns:
        summ[c] = pd.to_numeric(summ[c], errors="coerce")

# ensure numeric for details
for c in ["target_prop","sample_prop","abs_diff","diff_pp"]:
    if c in deta.columns:
        deta[c] = pd.to_numeric(deta[c], errors="coerce")

# -------------------------
# Pick tables to drill / summarize why
# -------------------------
def pick_tables_for_drilldown(cond_summary: pd.DataFrame,
                              tvd_mean_thr: float,
                              max_abs_dev_thr: float) -> pd.DataFrame:
    df = cond_summary.copy()
    df["flag_tvd_mean"] = df["tvd_mean"] > tvd_mean_thr
    df["flag_max_dev"]  = df["max_abs_dev_max"] > max_abs_dev_thr
    df["needs_drilldown"] = df[["flag_tvd_mean","flag_max_dev"]].any(axis=1)
    part_a = np.where(df["flag_tvd_mean"],  f"tvd_mean>{tvd_mean_thr}", "")
    part_b = np.where(df["flag_max_dev"],   f"max_abs_dev_max>{max_abs_dev_thr}", "")
    df["reason"] = [" & ".join([a,b]).strip(" & ") for a,b in zip(part_a, part_b)]
    return df

picks = pick_tables_for_drilldown(summ, TVD_MEAN_THR, MAX_ABS_DEV_THR)

# -------------------------
# Diagnostics at table-level (helps priority & action type)
# -------------------------
def _table_detail_diagnostics(details: pd.DataFrame, eps: float = 1e-9) -> pd.DataFrame:
    d = details.copy()
    d["delta"] = d["sample_prop"] - d["target_prop"]
    d["zero_coverage"]  = (d["target_prop"] > eps) & (d["sample_prop"] <= eps)
    d["only_in_sample"] = (d["target_prop"] <= eps) & (d["sample_prop"] >  eps)
    d["over"]  = (d["delta"] >  eps)
    d["under"] = (d["delta"] < -eps)
    d["sig"]   = (d["delta"].abs() > eps)

    grp = d.groupby("table", as_index=False).agg(
        n_rows=("table","size"),
        zero_cov=("zero_coverage","sum"),
        only_in=("only_in_sample","sum"),
        over_sig=("over","sum"),
        under_sig=("under","sum"),
        sig_cnt=("sig","sum"),
        mean_abs_diff=("abs_diff","mean"),
        max_abs_diff=("abs_diff","max"),
    )
    grp["zero_coverage_rate"]  = grp["zero_cov"] / grp["n_rows"].replace(0,np.nan)
    grp["only_in_sample_rate"] = grp["only_in"]  / grp["n_rows"].replace(0,np.nan)
    grp["direction_bias"] = np.where(
        grp["sig_cnt"]>0,
        np.maximum(grp["over_sig"], grp["under_sig"]) / grp["sig_cnt"],
        0.0
    )
    return grp[["table","n_rows","zero_coverage_rate","only_in_sample_rate","direction_bias",
                "mean_abs_diff","max_abs_diff"]]

diag = _table_detail_diagnostics(deta)

# merge preflight brief (status/issue)
pre_min = pre.groupby("table", as_index=False).agg(
    preflight_status=("status","first"),
    preflight_issue=("issue","first")
)

exec_summary = (picks.merge(pre_min, on="table", how="left")
                     .merge(diag, on="table", how="left")
               )

# Priority & suggested high-level action
def _priority_row(pre_status, tvd_mean, max_abs_dev):
    if str(pre_status).upper() == "FAIL":
        return "High"
    if (pd.notna(tvd_mean) and tvd_mean > 0.08) or (pd.notna(max_abs_dev) and max_abs_dev > 0.10):
        return "High"
    if (pd.notna(tvd_mean) and tvd_mean > 0.05) or (pd.notna(max_abs_dev) and max_abs_dev > 0.08):
        return "Medium"
    return "Low"

def _suggest_action(pre_status, zero_rate, only_rate, direction_bias, tvd_mean, max_abs_dev):
    if str(pre_status).upper() == "FAIL":
        return "fix_mapping_or_missing_columns"
    if pd.notna(only_rate) and only_rate >= 0.10:
        return "label_alignment_normalization"
    if pd.notna(zero_rate) and zero_rate >= 0.10:
        return "ensure_support_or_sampling_quotas_for_rare"
    if pd.notna(direction_bias) and direction_bias >= 0.70:
        return "genderwise_calibration_raking"
    if (pd.notna(tvd_mean) and tvd_mean > TVD_MEAN_THR) or (pd.notna(max_abs_dev) and max_abs_dev > MAX_ABS_DEV_THR):
        return "genderwise_calibration_raking"
    return "light_calibration_or_leave_as_is"

exec_summary["priority"] = [
    _priority_row(r.get("preflight_status"), r.get("tvd_mean"), r.get("max_abs_dev_max"))
    for _, r in exec_summary.iterrows()
]
exec_summary["suggested_action"] = [
    _suggest_action(
        r.get("preflight_status"),
        r.get("zero_coverage_rate"),
        r.get("only_in_sample_rate"),
        r.get("direction_bias"),
        r.get("tvd_mean"),
        r.get("max_abs_dev_max"),
    )
    for _, r in exec_summary.iterrows()
]

# reorder & save exec summary
cols = ["table","bn_feature_col","preflight_status","preflight_issue",
        "tvd_mean","max_abs_dev_max","needs_drilldown","reason",
        "zero_coverage_rate","only_in_sample_rate","direction_bias",
        "suggested_action","priority","notes"]
cols = [c for c in cols if c in exec_summary.columns]
exec_summary = exec_summary[cols].sort_values(["priority","tvd_mean","max_abs_dev_max"], ascending=[True, False, False])
exec_summary_path = os.path.join(OUT_DIR, "exec_summary.csv")
exec_summary.to_csv(exec_summary_path, index=False)
print(f"[OK] Wrote exec_summary to: {exec_summary_path}")
display(exec_summary.head(10))

# -------------------------
# Fine-grained modification plan (per table/gender/feature)
# -------------------------
# Add diagnostics to details
det = deta.copy()
det["delta"] = det["sample_prop"] - det["target_prop"]
det["sign"] = np.where(
    np.isclose(det["delta"], 0.0, atol=1e-12), "equal",
    np.where(det["delta"] > 0, "over", "under")
)
det["zero_coverage"]  = (det["target_prop"] > 0) & (det["sample_prop"] == 0)
det["only_in_sample"] = (det["target_prop"] == 0) & (det["sample_prop"] > 0)

# keep only tables that need drilldown
need_tbls = set(exec_summary.loc[exec_summary["needs_drilldown"]==True, "table"])
det = det[det["table"].isin(need_tbls)].copy()

# Helper: compute per-gender residual (sum of proposed deltas) and add a balancing row
def _add_residual_rows(df_gender: pd.DataFrame, table: str, bn_feature_col: str, gender: str):
    # Sum of suggested deltas (pp) for this (table, gender). We want it close to 0.
    total_pp = df_gender["suggested_delta_pp"].sum()
    if abs(total_pp) <= RESIDUAL_MIN_PP:
        return df_gender  # small enough; ignore
    # create a balancing row (no specific feature): executor should redistribute proportionally
    bal = {
        "table": table,
        "bn_feature_col": bn_feature_col,
        "gender": gender,
        "feature": "<redistribute>",
        "target_prop": np.nan,
        "sample_prop": np.nan,
        "diff_pp": np.nan,
        "abs_diff": np.nan,
        "issue_tag": "residual_balance",
        "rationale": f"Sum of suggested deltas = {total_pp:+.2f} pp; add balancing row to keep sum=0 within gender.",
        "suggested_delta_pp": -total_pp,   # negate to bring sum to zero
        "priority": "Medium",
        "note": "Redistribute this residual across non-critical categories."
    }
    return pd.concat([df_gender, pd.DataFrame([bal])], ignore_index=True)

# Build plan rows
plan_rows = []

for tbl, g_df in det.groupby("table"):
    # quick lookups
    bn_feature_col = (g_df["bn_feature_col"].dropna().astype(str).iloc[0]
                      if "bn_feature_col" in g_df.columns and g_df["bn_feature_col"].notna().any()
                      else "")
    # 1) label alignment issues first
    label_issues = g_df[(g_df["only_in_sample"]) | (g_df["zero_coverage"])].copy()
    for _, r in label_issues.iterrows():
        issue = "label_alignment" if r["only_in_sample"] else "missing_support"
        plan_rows.append({
            "table": tbl,
            "bn_feature_col": bn_feature_col,
            "gender": r["gender"],
            "feature": r["feature"],
            "target_prop": r["target_prop"],
            "sample_prop": r["sample_prop"],
            "diff_pp": r["diff_pp"],
            "abs_diff": r["abs_diff"],
            "issue_tag": issue,
            "rationale": "Category exists only on one side (enum/label mismatch or zero support). Fix labels/support before calibration.",
            "suggested_delta_pp": 0.0,   # alignment first; do not propose numeric tweak yet
            "priority": "High",
            "note": "Unify labels, dashes, spacing, and ensure category support in BN."
        })

    # 2) continuous calibration items (top |diff| per gender, excluding the alignment rows)
    cont = g_df[~(g_df["only_in_sample"] | g_df["zero_coverage"])].copy()
    # ignore tiny diffs
    cont = cont[cont["abs_diff"] >= MIN_ABS_DIFF_PCT].copy()

    for gender, gg in cont.groupby("gender"):
        # rank by absolute deviation
        gg2 = gg.sort_values("abs_diff", ascending=False).head(TOPK_PER_GENDER).copy()
        # propose delta = -diff_pp  (move sample towards target)
        # diff_pp = (sample - target) * 100
        gg2["suggested_delta_pp"] = -gg2["diff_pp"]
        gg2["issue_tag"] = np.where(
            (gg2["sign"] == "over"), "systematic_over_or_spike",
            np.where(gg2["sign"] == "under", "systematic_under_or_spike", "near_equal")
        )
        gg2["rationale"] = "Large absolute deviation within gender; nudge this category towards the target by -diff_pp."
        gg2["priority"]  = np.where(gg2["abs_diff"] >= 0.01, "High", "Medium")  # ≥1pp is High
        gg2["note"]      = "Apply within-gender normalization after per-category nudges."

        keep_cols = ["table","bn_feature_col","gender","feature",
                     "target_prop","sample_prop","diff_pp","abs_diff",
                     "issue_tag","rationale","suggested_delta_pp","priority","note"]
        gg2 = gg2[keep_cols]

        # add a residual balancing row per gender
        gg2 = _add_residual_rows(gg2, tbl, bn_feature_col, gender)
        plan_rows.extend(gg2.to_dict(orient="records"))

# Assemble plan
if len(plan_rows) == 0:
    modification_plan = pd.DataFrame(columns=[
        "table","bn_feature_col","gender","feature",
        "target_prop","sample_prop","diff_pp","abs_diff",
        "issue_tag","rationale","suggested_delta_pp","priority","note"
    ])
else:
    modification_plan = pd.DataFrame(plan_rows)

# Order rows for readability
order_cols = ["table","bn_feature_col","gender","feature",
              "target_prop","sample_prop","diff_pp","abs_diff",
              "issue_tag","rationale","suggested_delta_pp","priority","note"]
modification_plan = modification_plan[order_cols]

# Sort: High priority first, then by |suggested_delta_pp| desc
prio_order = pd.CategoricalDtype(categories=["High","Medium","Low"], ordered=True)
if "priority" in modification_plan.columns:
    modification_plan["priority"] = modification_plan["priority"].astype(prio_order)
modification_plan["abs_suggest_pp"] = modification_plan["suggested_delta_pp"].abs()
modification_plan = modification_plan.sort_values(
    by=["priority","table","gender","abs_suggest_pp"],
    ascending=[True, True, True, False]
).drop(columns=["abs_suggest_pp"])

# Save & preview
plan_path = os.path.join(OUT_DIR, "modification_plan.csv")
modification_plan.to_csv(plan_path, index=False)
print(f"[OK] Wrote modification_plan to: {plan_path}")
display(modification_plan.head(20))

[OK] Wrote exec_summary to: ./eval_outputs_v3/exec_summary.csv


,table,bn_feature_col,preflight_status,preflight_issue,tvd_mean,max_abs_dev_max,needs_drilldown,reason,zero_coverage_rate,only_in_sample_rate,direction_bias,suggested_action,priority,notes
4,sector_gender_transformed_v3.csv,Institutional_sector,OK,explicit TABLE_COLUMN_MAP used,0.274498,0.300602,True,tvd_mean>0.06 & max_abs_dev_max>0.08,0.0,0.400000,0.600000,label_alignment_normalization,High,dropped_unmapped=2; sample_cats=5
3,industry_gender_transformed_v3.csv,Industry,OK,explicit TABLE_COLUMN_MAP used,0.209199,0.078995,True,tvd_mean>0.06,0.0,0.076923,0.512821,genderwise_calibration_raking,High,dropped_unmapped=0; sample_cats=19
0,age_gender_transformed_v3.csv,Age,OK,explicit TABLE_COLUMN_MAP used,0.138987,0.054862,True,tvd_mean>0.06,0.0,0.000000,0.500000,genderwise_calibration_raking,High,dropped_unmapped=0; sample_cats=17
1,arrivalGroup_gender_transformed.csv,Arrival_group,OK,explicit TABLE_COLUMN_MAP used,0.102707,0.105229,True,tvd_mean>0.06 & max_abs_dev_max>0.08,0.0,0.000000,0.500000,genderwise_calibration_raking,High,dropped_unmapped=0; sample_cats=3
6,visaGroup_gender_transformed.csv,Visa_group,OK,explicit TABLE_COLUMN_MAP used,0.097575,0.062520,True,tvd_mean>0.06,0.0,0.000000,0.535714,genderwise_calibration_raking,High,dropped_unmapped=0; sample_cats=14
2,businessSize_gender_transformed.csv,Business_size,OK,explicit TABLE_COLUMN_MAP used,0.070009,0.072586,True,tvd_mean>0.06,0.0,0.000000,0.500000,genderwise_calibration_raking,Medium,dropped_unmapped=0; sample_cats=4
5,jobDuration_gender_transformed.csv,Job_duration,OK,explicit TABLE_COLUMN_MAP used,0.050681,0.052006,False,,0.0,0.000000,0.500000,light_calibration_or_leave_as_is,Medium,dropped_unmapped=0; sample_cats=4


[OK] Wrote modification_plan to: ./eval_outputs_v3/modification_plan.csv


,table,bn_feature_col,gender,feature,target_prop,sample_prop,diff_pp,abs_diff,issue_tag,rationale,suggested_delta_pp,priority,note
0,age_gender_transformed_v3.csv,Age,FEMALES,50 to 54 years,0.087030,0.032168,-5.486210,0.054862,systematic_under_or_spike,Large absolute deviation within gender; nudge ...,5.486210,High,Apply within-gender normalization after per-ca...
1,age_gender_transformed_v3.csv,Age,FEMALES,45 to 49 years,0.095059,0.041157,-5.390158,0.053902,systematic_under_or_spike,Large absolute deviation within gender; nudge ...,5.390158,High,Apply within-gender normalization after per-ca...
2,age_gender_transformed_v3.csv,Age,FEMALES,30 to 34 years,0.120086,0.172294,5.220827,0.052208,systematic_over_or_spike,Large absolute deviation within gender; nudge ...,-5.220827,High,Apply within-gender normalization after per-ca...
3,age_gender_transformed_v3.csv,Age,FEMALES,18 to 20 years,0.062384,0.086727,2.434244,0.024342,systematic_over_or_spike,Large absolute deviation within gender; nudge ...,-2.434244,High,Apply within-gender normalization after per-ca...
4,age_gender_transformed_v3.csv,Age,FEMALES,55 to 59 years,0.076093,0.054849,-2.124398,0.021244,systematic_under_or_spike,Large absolute deviation within gender; nudge ...,2.124398,High,Apply within-gender normalization after per-ca...
5,age_gender_transformed_v3.csv,Age,FEMALES,35 to 39 years,0.106617,0.122375,1.575743,0.015757,systematic_over_or_spike,Large absolute deviation within gender; nudge ...,-1.575743,High,Apply within-gender normalization after per-ca...
6,age_gender_transformed_v3.csv,Age,FEMALES,25 to 29 years,0.132089,0.144621,1.253161,0.012532,systematic_over_or_spike,Large absolute deviation within gender; nudge ...,-1.253161,High,Apply within-gender normalization after per-ca...
7,age_gender_transformed_v3.csv,Age,FEMALES,65 to 69 years,0.026140,0.038361,1.222118,0.012221,systematic_over_or_spike,Large absolute deviation within gender; nudge ...,-1.222118,High,Apply within-gender normalization after per-ca...
11,age_gender_transformed_v3.csv,Age,MALES,50 to 54 years,0.082365,0.134556,5.219088,0.052191,systematic_over_or_spike,Large absolute deviation within gender; nudge ...,-5.219088,High,Apply within-gender normalization after per-ca...
12,age_gender_transformed_v3.csv,Age,MALES,45 to 49 years,0.091555,0.142837,5.128215,0.051282,systematic_over_or_spike,Large absolute deviation within gender; nudge ...,-5.128215,High,Apply within-gender normalization after per-ca...


In [18]:
# === Consolidated Evaluation Report Generator ===
# Reads your existing outputs in OUT_DIR and emits:
#   - OUT_DIR/evaluation_result.md
#   - OUT_DIR/evaluation_result.tex
# Uses/creates exec_summary.csv & modification_plan.csv if needed.

import os
import numpy as np
import pandas as pd

OUT_DIR = "./eval_outputs_v3"   
TVD_MEAN_THR    = 0.06          
MAX_ABS_DEV_THR = 0.08          
ONLY_RATE_THR   = 0.10         
ZERO_RATE_THR   = 0.10      
BIAS_THR        = 0.70        
TOPK_TABLES     = 5            

def _must(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")
    return path

# --- Load base outputs
pre  = pd.read_csv(_must(os.path.join(OUT_DIR, "preflight_report.csv")))
summ = pd.read_csv(_must(os.path.join(OUT_DIR, "conditional_compare_summary.csv")))
deta = pd.read_csv(_must(os.path.join(OUT_DIR, "conditional_compare_details.csv")))

# Ensure numeric
for c in ["tvd_mean","tvd_max","hellinger_mean","hellinger_max","max_abs_dev_mean","max_abs_dev_max"]:
    if c in summ.columns:
        summ[c] = pd.to_numeric(summ[c], errors="coerce")
for c in ["target_prop","sample_prop","abs_diff","diff_pp"]:
    if c in deta.columns:
        deta[c] = pd.to_numeric(deta[c], errors="coerce")

exec_summary_path = os.path.join(OUT_DIR, "exec_summary.csv")
mod_plan_path     = os.path.join(OUT_DIR, "modification_plan.csv")

def _pick_tables_for_drilldown(df, tvd_thr, mad_thr):
    out = df.copy()
    out["flag_tvd_mean"] = out["tvd_mean"] > tvd_thr
    out["flag_max_dev"]  = out["max_abs_dev_max"] > mad_thr
    out["needs_drilldown"] = out[["flag_tvd_mean","flag_max_dev"]].any(axis=1)
    a = np.where(out["flag_tvd_mean"], f"tvd_mean>{tvd_thr}", "")
    b = np.where(out["flag_max_dev"],  f"max_abs_dev_max>{mad_thr}", "")
    out["reason"] = [" & ".join([x,y]).strip(" & ") for x,y in zip(a,b)]
    return out

def _table_detail_diag(details, eps=1e-9):
    d = details.copy()
    d["delta"] = d["sample_prop"] - d["target_prop"]
    d["zero_coverage"]   = (d["target_prop"] > eps) & (d["sample_prop"] <= eps)
    d["only_in_sample"]  = (d["target_prop"] <= eps) & (d["sample_prop"] >  eps)
    d["over"]  = (d["delta"] >  eps)
    d["under"] = (d["delta"] < -eps)
    d["sig"]   = (d["delta"].abs() > eps)
    g = d.groupby("table", as_index=False).agg(
        n_rows=("table","size"),
        zero_cov=("zero_coverage","sum"),
        only_in=("only_in_sample","sum"),
        over_sig=("over","sum"),
        under_sig=("under","sum"),
        sig_cnt=("sig","sum"),
        mean_abs_diff=("abs_diff","mean"),
        max_abs_diff=("abs_diff","max"),
    )
    g["zero_coverage_rate"]  = g["zero_cov"] / g["n_rows"].replace(0,np.nan)
    g["only_in_sample_rate"] = g["only_in"]  / g["n_rows"].replace(0,np.nan)
    g["direction_bias"] = np.where(
        g["sig_cnt"]>0,
        np.maximum(g["over_sig"], g["under_sig"]) / g["sig_cnt"],
        0.0
    )
    return g[["table","n_rows","zero_coverage_rate","only_in_sample_rate","direction_bias",
              "mean_abs_diff","max_abs_diff"]]

if os.path.exists(exec_summary_path) and os.path.exists(mod_plan_path):
    exec_summary = pd.read_csv(exec_summary_path)
    modification_plan = pd.read_csv(mod_plan_path)
else:
    picks = _pick_tables_for_drilldown(summ, TVD_MEAN_THR, MAX_ABS_DEV_THR)
    diag  = _table_detail_diag(deta)
    pre_min = pre.groupby("table", as_index=False).agg(
        preflight_status=("status","first"),
        preflight_issue=("issue","first")
    )
    exec_summary = (picks.merge(pre_min, on="table", how="left")
                         .merge(diag, on="table", how="left"))
    # 粗略建议
    def _priority(pre_status, tvd_mean, max_abs_dev):
        if str(pre_status).upper()=="FAIL": return "High"
        if (pd.notna(tvd_mean) and tvd_mean>0.08) or (pd.notna(max_abs_dev) and max_abs_dev>0.10): return "High"
        if (pd.notna(tvd_mean) and tvd_mean>0.05) or (pd.notna(max_abs_dev) and max_abs_dev>0.08): return "Medium"
        return "Low"
    def _suggest(pre_status, zero_rate, only_rate, bias, tvd_mean, max_abs_dev):
        if str(pre_status).upper()=="FAIL": return "fix_mapping_or_missing_columns"
        if pd.notna(only_rate) and only_rate>=ONLY_RATE_THR: return "label_alignment_normalization"
        if pd.notna(zero_rate) and zero_rate>=ZERO_RATE_THR: return "ensure_support_or_sampling_quotas_for_rare"
        if pd.notna(bias) and bias>=BIAS_THR: return "genderwise_calibration_raking"
        if (pd.notna(tvd_mean) and tvd_mean>TVD_MEAN_THR) or (pd.notna(max_abs_dev) and max_abs_dev>MAX_ABS_DEV_THR):
            return "genderwise_calibration_raking"
        return "light_calibration_or_leave_as_is"
    exec_summary["priority"] = [
        _priority(r.get("preflight_status"), r.get("tvd_mean"), r.get("max_abs_dev_max"))
        for _,r in exec_summary.iterrows()
    ]
    exec_summary["suggested_action"] = [
        _suggest(r.get("preflight_status"), r.get("zero_coverage_rate"), r.get("only_in_sample_rate"),
                 r.get("direction_bias"), r.get("tvd_mean"), r.get("max_abs_dev_max"))
        for _,r in exec_summary.iterrows()
    ]
    exec_cols = ["table","bn_feature_col","preflight_status","preflight_issue",
                 "tvd_mean","max_abs_dev_max","needs_drilldown","reason",
                 "zero_coverage_rate","only_in_sample_rate","direction_bias",
                 "suggested_action","priority","notes"]
    exec_cols = [c for c in exec_cols if c in exec_summary.columns]
    exec_summary = exec_summary[exec_cols].sort_values(["priority","tvd_mean","max_abs_dev_max"],
                                                       ascending=[True, False, False])
    exec_summary.to_csv(exec_summary_path, index=False)

    det = deta.copy()
    det["delta"] = det["sample_prop"] - det["target_prop"]
    det["sign"] = np.where(np.isclose(det["delta"],0.0,atol=1e-12),"equal",
                           np.where(det["delta"]>0,"over","under"))
    need_tbls = set(exec_summary.loc[exec_summary["needs_drilldown"]==True,"table"])
    det = det[det["table"].isin(need_tbls)].copy()
    rows=[]
    for (tbl, g), gg in det.groupby(["table","gender"]):
        bncol = (gg["bn_feature_col"].dropna().astype(str).iloc[0]
                 if "bn_feature_col" in gg.columns and gg["bn_feature_col"].notna().any() else "")
        top = gg.sort_values("abs_diff", ascending=False).head(10).copy()
        top["issue_tag"] = np.where(top["sign"]=="over","systematic_over_or_spike",
                             np.where(top["sign"]=="under","systematic_under_or_spike","near_equal"))
        top["suggested_delta_pp"] = -top["diff_pp"]  # 朝向target微调
        top["priority"] = np.where(top["abs_diff"]>=0.01,"High","Medium")  # >=1pp高优
        top["note"] = "Apply within-gender normalization after nudges."
        keep = ["table","bn_feature_col","gender","feature","target_prop","sample_prop",
                "diff_pp","abs_diff","issue_tag","suggested_delta_pp","priority","note"]
        top["bn_feature_col"]=bncol
        rows.append(top[keep])
    modification_plan = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=[
        "table","bn_feature_col","gender","feature","target_prop","sample_prop",
        "diff_pp","abs_diff","issue_tag","suggested_delta_pp","priority","note"
    ])
    modification_plan.to_csv(mod_plan_path, index=False)

# --- Aggregate evaluation headline
total_tables = len(summ)
flag_df = exec_summary[exec_summary.get("needs_drilldown", False)==True]
n_flagged  = len(flag_df)
overall_mean_tvd = float(np.nanmean(summ["tvd_mean"])) if "tvd_mean" in summ else np.nan
flag_mean_tvd    = float(np.nanmean(flag_df["tvd_mean"])) if "tvd_mean" in flag_df else np.nan

# Causes count
cnt_only  = int((exec_summary.get("only_in_sample_rate",0) >= ONLY_RATE_THR).sum())
cnt_zero  = int((exec_summary.get("zero_coverage_rate",0) >= ZERO_RATE_THR).sum())
cnt_bias  = int((exec_summary.get("direction_bias",0) >= BIAS_THR).sum())
cnt_tvd   = int(((exec_summary.get("tvd_mean",0) > TVD_MEAN_THR) & 
                 (exec_summary.get("only_in_sample_rate",0) < ONLY_RATE_THR) &
                 (exec_summary.get("zero_coverage_rate",0)  < ZERO_RATE_THR)).sum())

# Top tables view
top_tbls = (exec_summary.copy()
            .assign(tvd_mean=pd.to_numeric(exec_summary["tvd_mean"], errors="coerce"))
            .sort_values(["priority","tvd_mean","max_abs_dev_max"], ascending=[True, False, False])
            .head(TOPK_TABLES))

# --- Compose Markdown
md_lines = []
md_lines += [
f"# Evaluation Summary",
f"- Total tables: **{total_tables}**; flagged for drill-down: **{n_flagged}**.",
f"- Mean TVD (overall / flagged): **{overall_mean_tvd:.4f} / {flag_mean_tvd:.4f}**.",
f"- Thresholds used: TVD_mean>{TVD_MEAN_THR}, MaxAbsDev>{MAX_ABS_DEV_THR}, zero/only rates ≥{int(ONLY_RATE_THR*100)}%, direction bias ≥{int(BIAS_THR*100)}%.",
"",
"## Likely Causes (counts of affected tables)",
f"- Label alignment / enumeration (only-in-sample ≥{int(ONLY_RATE_THR*100)}%): **{cnt_only}**",
f"- Missing support / rare categories (zero-coverage ≥{int(ZERO_RATE_THR*100)}%): **{cnt_zero}**",
f"- Directional bias (≥{int(BIAS_THR*100)}% of nontrivial gaps move same way): **{cnt_bias}**",
f"- Global miscalibration (high TVD without coverage issues): **{cnt_tvd}**",
"",
"## Top Tables to Address",
]
for _, r in top_tbls.iterrows():
    md_lines.append(
        f"- **{r.get('table','?')}** | tvd_mean={r.get('tvd_mean',np.nan):.4f}, "
        f"max_abs_dev_max={r.get('max_abs_dev_max',np.nan):.4f} | reason={r.get('reason','')} | "
        f"suggested={r.get('suggested_action','')}, priority={r.get('priority','')}"
    )
md_lines += [
"",
"## Next Steps (from auto plan)",
f"- See `{os.path.basename(mod_plan_path)}` for per-(table, gender, feature) nudges (`suggested_delta_pp`, in pp).",
"- Process",
"",
"## Note on Income Granularity",
"- Targets with gender × income were marginalized over income to match gender-only comparisons. If large, consistent gaps persist across many tables, consider upgrading to P(c|g,i) for finer calibration.",
]
with open(os.path.join(OUT_DIR, "evaluation_result.md"), "w", encoding="utf-8") as f:
    f.write("\n".join(md_lines))

# --- Compose LaTeX snippet
def _latex_escape(s):
    if pd.isna(s): return ""
    return (str(s).replace("&","\\&").replace("%","\\%")
            .replace("_","\\_").replace("#","\\#").replace("$","\\$")
            .replace("{","\\{").replace("}","\\}").replace("~","\\textasciitilde{}")
            .replace("^","\\textasciicircum{}").replace("\\","\\textbackslash{}"))

latex = []
latex += [r"\subsubsection{Evaluation Summary (Auto-generated)}"]

only_pct = int(ONLY_RATE_THR*100)
bias_pct = int(BIAS_THR*100)

latex += [(
    rf"In total, {total_tables} tables were evaluated; {n_flagged} were flagged. "
    rf"Mean TVD (overall/flagged) = {overall_mean_tvd:.4f}/{flag_mean_tvd:.4f}. "
    rf"Thresholds: TVD$_{{mean}}$>{TVD_MEAN_THR}, MaxAbsDev>{MAX_ABS_DEV_THR}, "
    rf"zero/only rates $\ge {only_pct}\%$, directional bias $\ge {bias_pct}\%$."
)]
latex += [r"\paragraph{Likely Causes.} "
          rf"Label alignment/enum: {cnt_only}; Missing support: {cnt_zero}; "
          rf"Directional bias: {cnt_bias}; Global miscalibration: {cnt_tvd}."]
latex += [r"\begin{table}[h]\centering\small",
          r"\begin{tabular}{lcccccc}\toprule",
          r"Table & tvd\_mean & maxAbsDev & Reason & Suggested & Priority \\ \midrule"]
for _, r in top_tbls.iterrows():
    latex.append(" {} & {:.4f} & {:.4f} & {} & {} & {} \\\\".format(
        _latex_escape(r.get("table","?")),
        r.get("tvd_mean", np.nan),
        r.get("max_abs_dev_max", np.nan),
        _latex_escape(r.get("reason","")),
        _latex_escape(r.get("suggested_action","")),
        _latex_escape(r.get("priority",""))
    ))
latex += [r"\bottomrule\end{tabular}\caption{Top flagged tables and actions.}\end{table}",
          r"\paragraph{Next Steps.} See \texttt{"+_latex_escape(os.path.basename(mod_plan_path))+r"} for per-category adjustments (in pp). "
          r"Fix labels/support first, then apply within-gender calibration and re-run evaluation. "
          r"If persistent multi-table gaps remain, consider income-stratified comparison $P(c\mid g,i)$."]

with open(os.path.join(OUT_DIR, "evaluation_result.tex"), "w", encoding="utf-8") as f:
    f.write("\n".join(latex))

print("[OK] Wrote:", os.path.join(OUT_DIR, "evaluation_result.md"))
print("[OK] Wrote:", os.path.join(OUT_DIR, "evaluation_result.tex"))
print("[OK] Exec summary at:", exec_summary_path)
print("[OK] Modification plan at:", mod_plan_path)

[OK] Wrote: ./eval_outputs_v3/evaluation_result.md
[OK] Wrote: ./eval_outputs_v3/evaluation_result.tex
[OK] Exec summary at: ./eval_outputs_v3/exec_summary.csv
[OK] Modification plan at: ./eval_outputs_v3/modification_plan.csv
